In [1]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# 1. MOTOR DE NAVEGACIÓN (Configuración de Laboratorio)
def iniciar_navegador():
    options = Options()
    options.binary_location = "/usr/bin/google-chrome"
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# 2. CONFIGURACIÓN DEL GRUPO (Cada grupo modifica aquí)
URL_OBJETIVO = "https://books.toscrape.com/"
SELECTOR_CONTENEDOR = "article.product_pod"
SELECTOR_DATO_A = "h3 a"
SELECTOR_DATO_B = "p.price_color"

# 3. EJECUCIÓN DEL SCRAPING
driver = iniciar_navegador()
dataset_final = []

try:
    print("Conectando a la fuente de datos...")
    driver.get(URL_OBJETIVO)
    time.sleep(5)  # Tiempo de espera para carga de datos dinámicos

    elementos = driver.find_elements(By.CSS_SELECTOR, SELECTOR_CONTENEDOR)
    print(f"Se encontraron {len(elementos)} registros potenciales.")

    for item in elementos:
        try:
            columna_a = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_A).text
            columna_b = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_B).text

            # Limpieza básica de caracteres no numéricos (opcional según el proyecto)
            valor_limpio = "".join(filter(str.isdigit, columna_b))

            dataset_final.append({
                "variable_nombre": columna_a,
                "variable_valor": valor_limpio,
                "status": 1.0
            })
        except:
            continue

    # 4. SALIDA DE DATOS (Para análisis posterior)
    if dataset_final:
        df = pd.DataFrame(dataset_final)
        nombre_archivo = "datos_extraidos_grupo.csv"
        df.to_csv(nombre_archivo, index=False)
        print(f"Proceso exitoso. Archivo generado: {nombre_archivo}")
        print(df.head())
    else:
        print("No se capturaron datos. Revisar los selectores CSS.")

finally:
    driver.quit()
    print("Navegador cerrado.")


Conectando a la fuente de datos...
Se encontraron 20 registros potenciales.
Proceso exitoso. Archivo generado: datos_extraidos_grupo.csv
                variable_nombre variable_valor  status
0            A Light in the ...           5177     1.0
1            Tipping the Velvet           5374     1.0
2                    Soumission           5010     1.0
3                 Sharp Objects           4782     1.0
4  Sapiens: A Brief History ...           5423     1.0
Navegador cerrado.
